# Stage 6 — XGBoost Tweedie + Regional Feature

Adds a same-day regional rainfall feature and sweeps the Tweedie loss variance power, well-suited to zero-inflated rainfall data.

## 1. Imports

In [1]:
import pandas as pd, numpy as np, json, time, warnings
warnings.filterwarnings("ignore")
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

## 2. Load data & build the next-day rainfall target

In [2]:
t0 = time.time()
df = pd.read_csv("../Data/Processed/Processed.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["Station", "date"]).reset_index(drop=True)
df["Target_Rainfall"] = df.groupby("Station")["BMD_Rainfall"].shift(-1)
df["next_date"] = df.groupby("Station")["date"].shift(-1)
gap_days = (df["next_date"] - df["date"]).dt.days
valid = df["Target_Rainfall"].notna() & (gap_days == 1)
df = df[valid].drop(columns=["next_date"]).reset_index(drop=True)

## 3. Regional feature: same-day rainfall from the rest of the region

Maps each station to one of the 4 EDA regions, then computes each station's same-day rainfall **excluding itself** — avoiding target leakage.

In [3]:
REGION_MAP = {
    "Sylhet":"Hill_NE","Srimangal":"Hill_NE","Rangamati":"Hill_NE",
    "Chittagong":"Coastal","CoxsBazar":"Coastal","Teknaf":"Coastal","Sitakunda":"Coastal",
    "Sandwip":"Coastal","Kutubdia":"Coastal","Hatiya":"Coastal","Khepupara":"Coastal",
    "Patuakhali":"Coastal","Bhola":"Coastal","Barisal":"Coastal","M.court":"Coastal","Mongla":"Coastal",
    "Khulna":"Coastal","Satkhira":"Coastal",
    "Dhaka":"Central","Faridpur":"Central","Madaripur":"Central","Tangail":"Central",
    "Comilla":"Central","Chandpur":"Central","Feni":"Central",
    "Rajshahi":"Plains_N","Bogra":"Plains_N","Rangpur":"Plains_N","Dinajpur":"Plains_N",
    "sydpur":"Plains_N","Ishurdi":"Plains_N","Jessore":"Plains_N","chuadanga":"Plains_N","Mymensingh":"Plains_N",
}
df["Region"] = df["Station"].map(REGION_MAP)
region_day = df.groupby(["Region","date"])["BMD_Rainfall"].agg(["sum","count"]).reset_index()
region_day.columns = ["Region","date","region_sum","region_n"]
df = df.merge(region_day, on=["Region","date"], how="left")
df["Region_Rain_Today"] = (df["region_sum"] - df["BMD_Rainfall"]) / (df["region_n"]-1).clip(lower=1)

## 4. Locked chronological split (same cutoffs as Stage 1)

In [4]:
feature_cols = ["Year","Month","DOY","T2M","RH2M","PS","WS2M","CLOUD_AMT","ALLSKY_SFC_SW_DWN",
                "T2MDEW","Rain_lag1","Rain_lag2","Rain_7day_avg","Temp_lag1","Humidity_lag1",
                "Pressure_lag1","Station_Mean_Rainfall","Region_Rain_Today"]

train = df[df["Year"] <= 2007]; val = df[(df["Year"]>=2008)&(df["Year"]<=2011)]; test = df[df["Year"]>=2012]
X_train, y_train = train[feature_cols], train["Target_Rainfall"]
X_val, y_val = val[feature_cols], val["Target_Rainfall"]
X_test, y_test = test[feature_cols], test["Target_Rainfall"]

def score(name, pred):
    pred = np.clip(pred, 0, None)
    rmse = np.sqrt(mean_squared_error(y_test, pred)); mae = mean_absolute_error(y_test, pred); r2 = r2_score(y_test, pred)
    print(f"{name:42s} RMSE={rmse:7.3f}  MAE={mae:6.3f}  R2={r2:7.4f}  ({time.time()-t0:.1f}s)")
    return {"RMSE": float(rmse), "MAE": float(mae), "R2": float(r2)}

## 5. Sweep the Tweedie variance power

`p=1` behaves Poisson-like, `p=2` Gamma-like — rainfall (zero-inflated, continuous when positive) usually sits in the 1.1–1.9 range.

In [ ]:
results = {}
for power in [1.1, 1.3, 1.5, 1.7, 1.9]:
    m = xgb.XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.05, subsample=0.8,
                          colsample_bytree=0.8, reg_lambda=1.0, n_jobs=-1, random_state=42,
                          objective="reg:tweedie", tweedie_variance_power=power,
                          early_stopping_rounds=30, eval_metric="tweedie-nloglik@" + str(power))
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    results[f"Tweedie p={power}"] = score(f"XGBoost Tweedie (p={power})", m.predict(X_test))

XGBoost Tweedie (p=1.1)                    RMSE= 14.300  MAE= 5.831  R2= 0.4238  (27.2s)
XGBoost Tweedie (p=1.3)                    RMSE= 14.253  MAE= 5.721  R2= 0.4275  (56.7s)


## 6. Save results

In [ ]:
#--json.dump(results, open("results_tweedie.json", "w"), indent=2)
os.makedirs("../outputs", exist_ok=True)
pd.DataFrame(results).T.to_csv("../outputs/model-results/stage4_xgboost_tweedie_regional_results.csv")
print("done", time.time()-t0)